# Chosen Model Analysis
Post-selection analysis of the frozen 50/50 LGBM–Dynamic DeepSets ensemble. Completed outputs are loaded rather than recomputed. Component importance is compared by ranks.

In [ ]:
import os, sys, importlib, pickle
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR=Path('/content/drive/MyDrive/Colab Notebooks/FDS')
if not (PROJECT_DIR/'src').is_dir(): raise FileNotFoundError(PROJECT_DIR/'src')
os.chdir(PROJECT_DIR); sys.path.insert(0,str(PROJECT_DIR)) if str(PROJECT_DIR) not in sys.path else None
for name in tuple(sys.modules):
    if name=='src' or name.startswith('src.'): del sys.modules[name]
importlib.invalidate_caches(); import src
print(Path(src.__file__).resolve())

In [ ]:
from src.config import ExperimentConfig, UniverseConfig, FINAL_MODEL_ROSTER, FEATURES_40
from src.ensemble import ENSEMBLE_ID
CONFIG=ExperimentConfig(
 experiment_id='final_models_v1',project_dir=PROJECT_DIR,
 data_path=PROJECT_DIR/'jkp_USA_100chars_1980_2024.parquet',
 output_dir=PROJECT_DIR/'model_runs',selected_models=('LGBM_40','DEEPSET_40_DYNAMIC'),
 seed=42,use_gpu=False,universe=UniverseConfig(security_id_col='id'))
CONFIG.validate(); OUTPUT=CONFIG.run_dir/'chosen_model_analysis'/ENSEMBLE_ID
OUTPUT.mkdir(parents=True,exist_ok=True)

## Paired component comparisons and volatility regimes
Use the four checkboxes in the next cell to choose which component and metric comparisons to display. The complete four-row comparison is calculated once and retained so that Holm-adjusted p-values always use the same pre-specified family of tests. Changing the checkboxes does not recompute models or overwrite the complete saved table.

In [ ]:
from src.chosen_model_analysis import compare_chosen_to_components,build_volatility_regimes,analyze_regime_stability
paired_path=OUTPUT/'aligned_component_formal_comparisons.csv'
regime_path=OUTPUT/'regime_stability_summary.csv'
regimes=build_volatility_regimes(CONFIG)
paired_tests=pd.read_csv(paired_path) if paired_path.exists() else compare_chosen_to_components(CONFIG,ENSEMBLE_ID)
show_lgbm_comparison = True #@param {type:"boolean"}
show_deepsets_comparison = True #@param {type:"boolean"}
show_rank_ic = True #@param {type:"boolean"}
show_long_short_return = True #@param {type:"boolean"}
selected_components=[]
if show_lgbm_comparison: selected_components.append('LGBM_40')
if show_deepsets_comparison: selected_components.append('DEEPSET_40_DYNAMIC')
selected_metrics=[]
if show_rank_ic: selected_metrics.append('rank_ic')
if show_long_short_return: selected_metrics.append('long_short_return')
if not selected_components or not selected_metrics: raise ValueError('Select at least one component and one metric.')
selected_paired_tests=paired_tests.loc[paired_tests.component_model_id.isin(selected_components) & paired_tests.metric.isin(selected_metrics)].reset_index(drop=True)
regime_complete=regime_path.exists() and (OUTPUT/'monthly_regime_results.parquet').exists()
regime_summary=pd.read_csv(regime_path) if regime_complete else analyze_regime_stability(CONFIG,ENSEMBLE_ID,regimes)
display(selected_paired_tests); display(regime_summary)

## Exploratory comparisons between any models
This section is deliberately separate from the pre-specified evidence above. Enter one or more pairs as `MODEL_A,MODEL_B`, separated by semicolons. Results are always reported as Model A minus Model B. Running this cell never changes the formal comparison file or retrains a model.

In [ ]:
from src.chosen_model_analysis import compare_model_pairs
ALL_MODEL_IDS=(*FINAL_MODEL_ROSTER,ENSEMBLE_ID)
print('Available model IDs:')
print('\n'.join(f'  {model_id}' for model_id in ALL_MODEL_IDS))
PAIR_SPEC = "ENSEMBLE_LGBM40_DEEPSET40_DYNAMIC_50_50,LGBM_40; LGBM_40,DEEPSET_40_DYNAMIC" #@param {type:"string"}
compare_rank_ic = True #@param {type:"boolean"}
compare_long_short_return = True #@param {type:"boolean"}
save_exploratory_table = False #@param {type:"boolean"}
def parse_pair_spec(spec):
    pairs=[]
    for item in spec.split(';'):
        if not item.strip(): continue
        names=[name.strip() for name in item.split(',')]
        if len(names)!=2: raise ValueError(f'Use MODEL_A,MODEL_B for each pair; invalid entry: {item!r}')
        unknown=set(names)-set(ALL_MODEL_IDS)
        if unknown: raise ValueError(f'Unknown model ID(s): {sorted(unknown)}')
        pairs.append((names[0],names[1]))
    if not pairs: raise ValueError('Enter at least one model pair.')
    return pairs
exploratory_metrics=[]
if compare_rank_ic: exploratory_metrics.append('rank_ic')
if compare_long_short_return: exploratory_metrics.append('long_short_return')
exploratory_pairs=parse_pair_spec(PAIR_SPEC)
exploratory_comparison=compare_model_pairs(CONFIG,exploratory_pairs,metrics=exploratory_metrics)
if save_exploratory_table:
    from src.artifacts import write_csv_atomic
    write_csv_atomic(exploratory_comparison,OUTPUT/'exploratory_pairwise_comparisons.csv')
display(exploratory_comparison.drop(columns='interpretation'))
print('\nInterpretation')
for text in exploratory_comparison.interpretation: print(f'• {text}')

## Prepare the exact OOS feature panel only when importance outputs are incomplete

In [ ]:
SHAP_PATH=OUTPUT/'lgbm_shap_importance.csv'
DEEP_PATH=OUTPUT/'deepset_permutation_importance.csv'
def complete_shap(path):
    if not path.exists(): return False
    result=pd.read_csv(path); required={'regime','feature','mean_abs_shap','rank'}
    return required.issubset(result.columns) and set(result.feature)==set(FEATURES_40) and set(result.regime)=={'ALL','HIGH_VOL','LOW_VOL'}
def complete_deep(path):
    if not path.exists(): return False
    result=pd.read_csv(path); required={'characteristic','all_importance','high_vol_importance','low_vol_importance','all_rank','high_vol_rank','low_vol_rank'}
    return required.issubset(result.columns) and set(result.characteristic)==set(FEATURES_40)
need_shap=not complete_shap(SHAP_PATH); need_deep=not complete_deep(DEEP_PATH)
data=None
if need_shap or need_deep:
    from src.data import load_and_prepare_panel
    from src.models import MODEL_FEATURES,MODEL_REGISTRY,build_deepset_core
    chosen=pd.read_parquet(CONFIG.run_dir/'predictions'/f'{ENSEMBLE_ID}.parquet')
    panel,_=load_and_prepare_panel(CONFIG,rank_features=FEATURES_40,include_feature40_lag1=True)
    required=sorted(set(MODEL_FEATURES['LGBM_40'])|set(MODEL_FEATURES['DEEPSET_40_DYNAMIC']))
    data=chosen[['eom','security_id','test_year','y_true']].merge(
        panel[['eom','security_id',*required]],on=['eom','security_id'],how='left',validate='one_to_one')
    if data[required].isna().all(axis=1).any(): raise RuntimeError('OOS feature merge failed')
    print('Importance rows:',len(data))
else:
    print('Complete importance outputs found; panel preparation skipped.')

## LightGBM pooled OOS TreeSHAP
Absolute TreeSHAP contributions are accumulated by annual OOS refit and pooled using observation counts. Annual checkpoints make this section resumable.

In [ ]:
from src.artifacts import write_csv_atomic
from src.chosen_model_analysis import summarize_lgbm_shap
SHAP_CHECKPOINT=OUTPUT/'lgbm_shap_annual_checkpoint.csv'
if not need_shap:
    lgbm_shap=pd.read_csv(SHAP_PATH)
else:
    lgbm_features=list(MODEL_FEATURES['LGBM_40']); regime_map=regimes.set_index('eom')['regime']
    annual=pd.read_csv(SHAP_CHECKPOINT) if SHAP_CHECKPOINT.exists() else pd.DataFrame()
    completed=set(annual.test_year.astype(int)) if not annual.empty else set()
    parts=[] if annual.empty else [annual]
    for year,year_data in data.groupby('test_year',sort=True):
        if int(year) in completed: continue
        with (CONFIG.run_dir/'models'/'LGBM_40'/f'refit_{int(year)}'/'model.bin').open('rb') as handle: model=pickle.load(handle)
        contribution=np.asarray(model.predict(year_data[lgbm_features].to_numpy(np.float32),pred_contrib=True))
        shap_values=contribution[:,:-1]; reconstructed=contribution.sum(axis=1)
        saved=pd.read_parquet(CONFIG.run_dir/'predictions'/'LGBM_40.parquet',filters=[('test_year','==',int(year))])
        aligned=year_data[['eom','security_id']].assign(reconstructed=reconstructed).merge(
            saved[['eom','security_id','y_pred']],on=['eom','security_id'],validate='one_to_one')
        if not np.allclose(aligned.reconstructed,aligned.y_pred,rtol=1e-5,atol=1e-7): raise RuntimeError(f'LGBM reconstruction failed for {year}')
        labels=year_data.eom.map(regime_map).to_numpy(); year_rows=[]
        for label,mask in [('ALL',np.ones(len(year_data),bool)),('HIGH_VOL',labels=='HIGH_VOL'),('LOW_VOL',labels=='LOW_VOL')]:
            if mask.any():
                year_rows.extend({'test_year':int(year),'regime':label,'feature':feature,'sum_abs_shap':float(value),'n_observations':int(mask.sum())} for feature,value in zip(lgbm_features,np.abs(shap_values[mask]).sum(axis=0)))
        parts.append(pd.DataFrame(year_rows)); write_csv_atomic(pd.concat(parts,ignore_index=True),SHAP_CHECKPOINT)
    lgbm_shap=summarize_lgbm_shap(pd.concat(parts,ignore_index=True))
    write_csv_atomic(lgbm_shap,SHAP_PATH)
lgbm_shap.sort_values(['regime','rank']).groupby('regime').head(10)

## Dynamic DeepSets grouped permutation importance
Each characteristic’s current, lag-1 and velocity coordinates are shuffled jointly within month and lag-availability stratum. Three fixed-seed repetitions are used. Completed characteristics are checkpointed.

In [ ]:
if not need_deep:
    deepset_importance=pd.read_csv(DEEP_PATH)
else:
    import torch
    features=list(MODEL_FEATURES['DEEPSET_40_DYNAMIC']); device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); cache={}
    def deepset_predict(frame):
        out=np.empty(len(frame),dtype=float)
        for year,index in frame.groupby('test_year',sort=True).indices.items():
            positions=np.asarray(index,dtype=np.int64); yf=frame.iloc[positions].reset_index(drop=True)
            if int(year) not in cache:
                model=build_deepset_core(len(features),MODEL_REGISTRY['DEEPSET_40_DYNAMIC'].params).to(device)
                state=torch.load(CONFIG.run_dir/'models'/'DEEPSET_40_DYNAMIC'/f'refit_{int(year)}'/'model.bin',map_location=device)
                model.load_state_dict(state); model.eval(); cache[int(year)]=model
            with torch.inference_mode():
                for _,month_index in yf.groupby('eom',sort=True).indices.items():
                    mp=np.asarray(month_index,dtype=np.int64); x=torch.from_numpy(yf.iloc[mp][features].to_numpy(np.float32)).to(device)
                    out[positions[mp]]=cache[int(year)](x).cpu().numpy()
        return out
    saved=pd.read_parquet(CONFIG.run_dir/'predictions'/'DEEPSET_40_DYNAMIC.parquet')[['eom','security_id','test_year','y_pred']]
    reconstructed=data[['eom','security_id','test_year']].assign(reconstructed=deepset_predict(data)).merge(
        saved,on=['eom','security_id','test_year'],validate='one_to_one')
    if not np.allclose(reconstructed.reconstructed,reconstructed.y_pred,rtol=1e-5,atol=1e-7): raise RuntimeError('DeepSets reconstruction differs from saved OOS predictions')
    from src.chosen_model_analysis import analyze_characteristic_stability
    deepset_importance=analyze_characteristic_stability(CONFIG,ENSEMBLE_ID,data=data,predict_fn=deepset_predict,regimes=regimes,n_repeats=3,seed=42)
deepset_importance.head(10)

## Compare component importance ranks

In [ ]:
RANK_PATH=OUTPUT/'component_importance_rank_comparison.csv'
if RANK_PATH.exists():
    rank_comparison=pd.read_csv(RANK_PATH,index_col=0)
else:
    lgbm_ranks=lgbm_shap.pivot(index='feature',columns='regime',values='rank').add_prefix('lgbm_rank_')
    deep=deepset_importance.set_index('characteristic')[['all_rank','high_vol_rank','low_vol_rank']].add_prefix('deepset_')
    rank_comparison=lgbm_ranks.join(deep,how='outer'); rank_comparison.to_csv(RANK_PATH)
rank_comparison.sort_values('deepset_all_rank').head(15)

## Portfolio-construction robustness

In [ ]:
from src.chosen_model_analysis import analyze_portfolio_robustness
ROBUST_PATH=OUTPUT/'portfolio_robustness_summary.csv'
robust_complete=ROBUST_PATH.exists() and (OUTPUT/'portfolio_robustness_monthly.parquet').exists()
portfolio_robustness=pd.read_csv(ROBUST_PATH) if robust_complete else analyze_portfolio_robustness(CONFIG,ENSEMBLE_ID)
portfolio_robustness

## Final artifact audit

In [ ]:
from src.post_train_audit import run_post_train_audit
audit=run_post_train_audit(CONFIG,standalone_model_ids=FINAL_MODEL_ROSTER,chosen_model_id=ENSEMBLE_ID)
failed=audit.loc[~audit.passed]
if not failed.empty: display(failed); raise RuntimeError(f'Final audit failed {len(failed)} checks')
print(f'FINAL AUDIT PASS: {len(audit)} checks')